In [ ]:
from src.agent import *
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np

model_id = "qwen/qwen2.5-0.5B-instruct"

questionAB = "Between Red and Blue which is your favorite?"
questionBA = "Between Blue and Red which is your favorite?"
labels = ['Red', 'Blue']
global model
global tokenizer
model, tokenizer = None, None

def load_model_cache(model_class):
    global model
    global tokenizer    
    if not model or not tokenizer:
        print("Did not find cache, loading model:")
        model, tokenizer = HFAgent._load_model_and_tokenizer(model_id)
    return model_class.with_loaded_model(model, tokenizer)

/home/lotanamit/miniconda3/envs/whatdo-llms-want/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Examining the chat template implementation

### lotan V0

my original version

In [16]:
lotan_v0 = load_model_cache(LotanHFAgentV0)

Did not find cache, loading model:
Loading qwen/qwen2.5-0.5B-instruct...


No GPU found. Running on CPU (not recommended for large models).


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1109.70it/s]


In [17]:
lotan_v0.query(questionAB, labels)

query: prompt=Between Red and Blue which is your favorite?, labels=['Red', 'Blue'].
chat template=<|im_start|>system
You are a helpful assistant. Answer shortly with only your choice with no explanation.

<|im_end|>
<|im_start|>user
Between Red and Blue which is your favorite?<|im_end|>
<|im_start|>assistant

label='Red'
input_ids=tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
          21806,  19620,    448,   1172,    697,   5754,    448,    902,  16148,
            382, 151645,    198, 151644,    872,    198,  25477,   3731,    323,
           8697,    892,    374,    697,   6930,     30, 151645,    198, 151644,
          77091,    198,   3731]])
prompt_ids=tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
          21806,  19620,    448,   1172,    697,   5754,    448,    902,  16148,
            382, 151645,    198, 151644,    872,    198,  25477,   3731,    323,
           8697,    892,    374,    697,   6930,     3

[-12.24665641784668, -11.455734252929688]

### lotan V1

encoding the labels and prompt seperatly

In [18]:
lotan_v1 = load_model_cache(LotanHFAgentV1)

In [19]:
lotan_v1.query(questionAB, labels)

query: prompt=Between Red and Blue which is your favorite?, labels=['Red', 'Blue'].
prompt='<|im_start|>system\nYou are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n<|im_end|>\n<|im_start|>user\nBetween Red and Blue which is your favorite?<|im_end|>\n<|im_start|>assistant\n'
prompt_ids=tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
          21806,  19620,    448,   1172,    697,   5754,    448,    902,  16148,
            382, 151645,    198, 151644,    872,    198,  25477,   3731,    323,
           8697,    892,    374,    697,   6930,     30, 151645,    198, 151644,
          77091,    198]])
label='Red'
label_ids=tensor([[6033]])
input_ids=tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
          21806,  19620,    448,   1172,    697,   5754,    448,    902,  16148,
            382, 151645,    198, 151644,    872,    198,  25477,   3731,    323,
           8697,    892,    374, 

[-2.6556591987609863, -2.996880054473877]

### Itay v0

In [21]:
itay_v0 = load_model_cache(ItayHFAgentV0)

In [22]:
question_converted = itay_v0.convert_to_chat_format(questionAB)
itay_v0.query([question_converted], labels)

input_with_answers=['<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nYou are a helpful assistant. Answer shortly with only your choice with no explanation.\n\nBetween Red and Blue which is your favorite?<|im_end|>\n<|im_start|>assistant\nRed', '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nYou are a helpful assistant. Answer shortly with only your choice with no explanation.\n\nBetween Red and Blue which is your favorite?<|im_end|>\n<|im_start|>assistant\nBlue']
labels_tokens=[[6033], [10331]]
labels_tokens=[6033, 10331]
input_enc={'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,   2610,    525,    264,
          10950,  17847,     13,  21806,  19620,    448,   1172,

tensor([-3.8136, -4.7460], grad_fn=<DiagonalBackward0_copy>)

### itay v1

In [23]:
itay_v1 = load_model_cache(ItayHFAgentV1)

In [24]:
question_converted = itay_v1.convert_to_chat_format(questionAB)
itay_v1.query([question_converted], labels)

input_with_answers=['<|im_start|>system\nYou are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n<|im_end|>\n<|im_start|>user\nBetween Red and Blue which is your favorite?<|im_end|>\n<|im_start|>assistant\nRed', '<|im_start|>system\nYou are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n<|im_end|>\n<|im_start|>user\nBetween Red and Blue which is your favorite?<|im_end|>\n<|im_start|>assistant\nBlue']
labels_tokens=[[6033], [10331]]
labels_tokens=[6033, 10331]
input_enc={'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
          21806,  19620,    448,   1172,    697,   5754,    448,    902,  16148,
            382, 151645,    198, 151644,    872,    198,  25477,   3731,    323,
           8697,    892,    374,    697,   6930,     30, 151645,    198, 151644,
          77091,    198,   6033],
        [151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
    

tensor([-2.6557, -2.9969], grad_fn=<DiagonalBackward0_copy>)

### lotan v1 - options prompt

In [25]:
lotan_v1 = load_model_cache(LotanHFAgentV1)

question2AB = "Which is better?"
labels = ["Option A", "Option B"]
prompt = "Choose between:\n" + \
"Option A - $4000 with a 20% chance, $0 with an 80% chance.\n" + \
"Option B - $3000 with a 25% chance, $0 with a 75% chance.\n" + \
"What is your choice?\n" + "Answer:"
print(prompt)
lotan_v1.query(prompt, labels)

Choose between:
Option A - $4000 with a 20% chance, $0 with an 80% chance.
Option B - $3000 with a 25% chance, $0 with a 75% chance.
What is your choice?
Answer:
query: prompt=Choose between:
Option A - $4000 with a 20% chance, $0 with an 80% chance.
Option B - $3000 with a 25% chance, $0 with a 75% chance.
What is your choice?
Answer:, labels=['Option A', 'Option B'].
prompt='<|im_start|>system\nYou are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n<|im_end|>\n<|im_start|>user\nChoose between:\nOption A - $4000 with a 20% chance, $0 with an 80% chance.\nOption B - $3000 with a 25% chance, $0 with a 75% chance.\nWhat is your choice?\nAnswer:<|im_end|>\n<|im_start|>assistant\n'
prompt_ids=tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
          21806,  19620,    448,   1172,    697,   5754,    448,    902,  16148,
            382, 151645,    198, 151644,    872,    198,  24051,   1948,    510,
           5341,    362,

[-1.0461571216583252, -0.4495994448661804]

In [26]:
prompt_flipped = "Choose between:\n" + \
"Option A - $3000 with a 25% chance, $0 with a 75% chance.\n" + \
"Option B - $4000 with a 20% chance, $0 with an 80% chance.\n" + \
"What is your choice?\n" + "Answer:"
print(prompt)
lotan_v1.query(prompt_flipped, labels)

Choose between:
Option A - $4000 with a 20% chance, $0 with an 80% chance.
Option B - $3000 with a 25% chance, $0 with a 75% chance.
What is your choice?
Answer:
query: prompt=Choose between:
Option A - $3000 with a 25% chance, $0 with a 75% chance.
Option B - $4000 with a 20% chance, $0 with an 80% chance.
What is your choice?
Answer:, labels=['Option A', 'Option B'].
prompt='<|im_start|>system\nYou are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n<|im_end|>\n<|im_start|>user\nChoose between:\nOption A - $3000 with a 25% chance, $0 with a 75% chance.\nOption B - $4000 with a 20% chance, $0 with an 80% chance.\nWhat is your choice?\nAnswer:<|im_end|>\n<|im_start|>assistant\n'
prompt_ids=tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
          21806,  19620,    448,   1172,    697,   5754,    448,    902,  16148,
            382, 151645,    198, 151644,    872,    198,  24051,   1948,    510,
           5341,    362,

[-1.093210220336914, -0.424495667219162]

### itay v1 - options prompt

In [27]:
prompt_chat = itay_v1.convert_to_chat_format(prompt)
itay_v1.query([prompt_chat], labels)

input_with_answers=['<|im_start|>system\nYou are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n<|im_end|>\n<|im_start|>user\nChoose between:\nOption A - $4000 with a 20% chance, $0 with an 80% chance.\nOption B - $3000 with a 25% chance, $0 with a 75% chance.\nWhat is your choice?\nAnswer:<|im_end|>\n<|im_start|>assistant\nOption A', '<|im_start|>system\nYou are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n<|im_end|>\n<|im_start|>user\nChoose between:\nOption A - $4000 with a 20% chance, $0 with an 80% chance.\nOption B - $3000 with a 25% chance, $0 with a 75% chance.\nWhat is your choice?\nAnswer:<|im_end|>\n<|im_start|>assistant\nOption B']
labels_tokens=[[5341, 362], [5341, 425]]
labels_tokens=[362, 425]
input_enc={'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
          21806,  19620,    448,   1172,    697,   5754,    448,    902,  16148,
            382, 151645,

tensor([-1.0392, -0.4427], grad_fn=<DiagonalBackward0_copy>)

In [18]:
import numpy as np

L = np.array([-1.0461571216583252, -0.4495994448661804])
I = np.array([-1.0392, -0.4427])

print(L[0] - I[0])
print(L[1] - I[1])

denom_L = (np.exp(L)).sum()
norm_L = np.exp(L) / denom_L

denom_I = (np.exp(I)).sum()
norm_I = np.exp(I) / denom_I

print(f"{norm_L=}")
print(f"{norm_I=}")

-0.006957121658325294
-0.0068994448661804375
norm_L=array([0.35513164, 0.64486836])
norm_I=array([0.35514485, 0.64485515])


In [31]:
lotan_v1 = load_model_cache(LotanHFAgentV1)

question2AB = "Which is better?"
labels_12 = ["Option 1", "Option 2"]
prompt3 = "Choose between:\n" + \
"Option 1 - $4000 with a 20% chance, $0 with an 80% chance.\n" + \
"Option 2 - $3000 with a 25% chance, $0 with a 75% chance.\n" + \
"What is your choice?\n" + "Answer:"
print(prompt3)
lotan_v1_prompt3_scores = lotan_v1.query(prompt3, labels_12)

Choose between:
Option 1 - $4000 with a 20% chance, $0 with an 80% chance.
Option 2 - $3000 with a 25% chance, $0 with a 75% chance.
What is your choice?
Answer:
query: prompt=Choose between:
Option 1 - $4000 with a 20% chance, $0 with an 80% chance.
Option 2 - $3000 with a 25% chance, $0 with a 75% chance.
What is your choice?
Answer:, labels=['Option 1', 'Option 2'].
prompt='<|im_start|>system\nYou are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n<|im_end|>\n<|im_start|>user\nChoose between:\nOption 1 - $4000 with a 20% chance, $0 with an 80% chance.\nOption 2 - $3000 with a 25% chance, $0 with a 75% chance.\nWhat is your choice?\nAnswer:<|im_end|>\n<|im_start|>assistant\n'
prompt_ids=tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
          21806,  19620,    448,   1172,    697,   5754,    448,    902,  16148,
            382, 151645,    198, 151644,    872,    198,  24051,   1948,    510,
           5341,    220,

In [32]:
prompt3_chat = itay_v1.convert_to_chat_format(prompt3)
itay_v1_prompt3_scores = itay_v1.query([prompt3_chat], labels_12)

input_with_answers=['<|im_start|>system\nYou are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n<|im_end|>\n<|im_start|>user\nChoose between:\nOption 1 - $4000 with a 20% chance, $0 with an 80% chance.\nOption 2 - $3000 with a 25% chance, $0 with a 75% chance.\nWhat is your choice?\nAnswer:<|im_end|>\n<|im_start|>assistant\nOption 1', '<|im_start|>system\nYou are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n<|im_end|>\n<|im_start|>user\nChoose between:\nOption 1 - $4000 with a 20% chance, $0 with an 80% chance.\nOption 2 - $3000 with a 25% chance, $0 with a 75% chance.\nWhat is your choice?\nAnswer:<|im_end|>\n<|im_start|>assistant\nOption 2']
labels_tokens=[[5341, 220, 16], [5341, 220, 17]]
labels_tokens=[16, 17]
input_enc={'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
          21806,  19620,    448,   1172,    697,   5754,    448,    902,  16148,
            382, 1

In [33]:
itay_v1_prompt3_scores

tensor([-1.2041, -0.3613], grad_fn=<DiagonalBackward0_copy>)

In [34]:
import numpy as np

L = np.array(lotan_v1_prompt3_scores)
I = np.array(itay_v1_prompt3_scores.detach().tolist())

print(L[0] - I[0])
print(L[1] - I[1])

denom_L = (np.exp(L)).sum()
norm_L = np.exp(L) / denom_L

denom_I = (np.exp(I)).sum()
norm_I = np.exp(I) / denom_I

print(f"{norm_L=}")
print(f"{norm_I=}")

-0.005021333694458008
-0.0050144195556640625
norm_L=array([0.30095901, 0.69904099])
norm_I=array([0.30096047, 0.69903953])


### lotan v2

In [4]:
lotan_v2 = load_model_cache(LotanHFAgentV2)

In [3]:
labels_12 = ["1", "2"]
prompt3 = "Choose between:\n" + \
"Option 1 - $4000 with a 20% chance, $0 with an 80% chance.\n" + \
"Option 2 - $3000 with a 25% chance, $0 with a 75% chance.\n" + \
"What is your choice?\n" + "Answer:"

In [ ]:
lotan_v2_prompt3_scores = lotan_v2.query(prompt3, labels_12, prefill="Option ")
lotan_v2_prompt3_scores

query: prompt=Choose between:
Option 1 - $4000 with a 20% chance, $0 with an 80% chance.
Option 2 - $3000 with a 25% chance, $0 with a 75% chance.
What is your choice?
Answer:, labels=['1', '2'], prefill='Option '.
prompt_with_prefill=<|im_start|>system
You are a helpful assistant. Answer shortly with only your choice with no explanation.

<|im_end|>
<|im_start|>user
Choose between:
Option 1 - $4000 with a 20% chance, $0 with an 80% chance.
Option 2 - $3000 with a 25% chance, $0 with a 75% chance.
What is your choice?
Answer:<|im_end|>
<|im_start|>assistant
Option 
prompt_ids=tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
          21806,  19620,    448,   1172,    697,   5754,    448,    902,  16148,
            382, 151645,    198, 151644,    872,    198,  24051,   1948,    510,
           5341,    220,     16,    481,    400,     19,     15,     15,     15,
            448,    264,    220,     17,     15,      4,   6012,     11,    400,
            

[-1.2040797472000122, -0.36134448647499084]

### lotan v3

In [4]:
lotan_v3 = load_model_cache(LotanHFAgentV3)

lotan_v3_prompt3_scores = lotan_v3.query(prompt3, labels_12, prefill="Option ")
lotan_v3_prompt3_scores

[-1.2040756940841675, -0.3613473176956177]

In [8]:
lotan_v3_prompt3_scores_normalized = lotan_v3.query(prompt3, labels_12, prefill="Option ", normalize="Answer: ")
lotan_v3_prompt3_scores_normalized

[0.02636253833770752, 0.6996482610702515]

In [12]:

a = np.array(lotan_v3_prompt3_scores)
b = np.array(lotan_v3_prompt3_scores_normalized)

a_ = np.exp(a) / np.exp(a).sum()
b_ = np.exp(b) / np.exp(b).sum()

print(a_)
print(b_)

[0.30096047 0.69903953]
[0.3377615 0.6622385]


In [14]:
prompt3_flipped = "Choose between:\n" + \
"Option 1 - $3000 with a 25% chance, $0 with a 75% chance.\n" + \
"Option 2 - $4000 with a 20% chance, $0 with an 80% chance.\n" + \
"What is your choice?\n" + "Answer:"

lotan_v3_prompt3_flipped_scores = lotan_v3.query(prompt3_flipped, labels_12, prefill="Option ")
lotan_v3_prompt3_flipped_scores_normalized = lotan_v3.query(prompt3_flipped, labels_12, prefill="Option ", normalize="Answer: ")

a = np.array(lotan_v3_prompt3_flipped_scores)
b = np.array(lotan_v3_prompt3_flipped_scores_normalized)
print("a: ",a)
print("b: ",b)

a_ = np.exp(a) / np.exp(a).sum()
b_ = np.exp(b) / np.exp(b).sum()

print("a_norm: ", a_)
print("b norm: ", b_)

a:  [-1.3629899  -0.29899463]
b:  [-0.13255167  0.76200095]
a_norm:  [0.25654669 0.74345331]
b norm:  [0.29017122 0.70982878]


In [15]:
prompt3_AB = "Choose between:\n" + \
"Option A - $3000 with a 25% chance, $0 with a 75% chance.\n" + \
"Option B - $4000 with a 20% chance, $0 with an 80% chance.\n" + \
"What is your choice?\n" + "Answer:"
labels_AB = ["A", "B"]
lotan_v3_prompt3_AB_scores = lotan_v3.query(prompt3_AB, labels_AB, prefill="Option ")
lotan_v3_prompt3_AB_scores_normalized = lotan_v3.query(prompt3_AB, labels_AB, prefill="Option ", normalize="Answer: ")

a = np.array(lotan_v3_prompt3_AB_scores)
b = np.array(lotan_v3_prompt3_AB_scores_normalized)
print("a: ",a)
print("b: ",b)

a_ = np.exp(a) / np.exp(a).sum()
b_ = np.exp(b) / np.exp(b).sum()

print("a_norm: ", a_)
print("b norm: ", b_)

a:  [-15.7919569  -13.37200069]
b:  [2.30892467 3.9274931 ]
a_norm:  [0.08166354 0.91833646]
b norm:  [0.1654024 0.8345976]


In [11]:
s = "s = {A}"
s.format(A='d')
s

's = {A}'